In [4]:
"""Build `waiter_level_features.parquet` using client scores from `client_anomaly_scores.csv`.

When `waiter_week_anomaly_scores.csv` exists at the project root (same as `models/waiter_week_models.py`),
adds `share_anomaly_weeks_iso`, `share_anomaly_weeks_ocsvm`, `share_anomaly_weeks_lof` using the same
logic as `models/lala.ipynb` (global score percentiles, flag > 0.95, share over weeks with 3+ distinct weeks),
and `avg_anomaly_score_iso` / `avg_anomaly_score_ocsvm` / `avg_anomaly_score_lof` (mean waiter-week scores per waiter).
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd()
ROOT = _here if (_here / "config.py").exists() else _here.parent
if not (ROOT / "config.py").exists():
    raise FileNotFoundError(f"Expected config.py under {ROOT}; run from project root or parquet/")

sys.path.insert(0, str(ROOT))

from config import DATA_PATH, FRAUD_IDS


def _share_anomaly_weeks_from_waiter_week_scores(
    ww: pd.DataFrame,
    percentile_threshold: float = 0.95,
    min_distinct_weeks: int = 3,
) -> pd.DataFrame:
    """
    Match models/lala.ipynb: rank anomaly scores globally across waiter-weeks, flag
    percentile > threshold, then share_anomaly_weeks_* = sum(flags) / n_distinct weeks
    for waiters with more than 2 distinct weeks (else NaN). Also returns mean iso/ocsvm/lof
    scores across that waiter's waiter-week rows.
    """
    d = ww.copy()
    score_cols = ["iso_score", "ocsvm_score", "lof_score"]
    for col in score_cols:
        d[f"{col}_percentile"] = d[col].rank(pct=True, method="average")
    perc_cols = ["iso_score_percentile", "ocsvm_score_percentile", "lof_score_percentile"]
    for pc in perc_cols:
        d[f"anomaly_{pc}"] = d[pc] > percentile_threshold

    agg = d.groupby("waiter_id", as_index=False).agg(
        anomaly_weeks_iso=("anomaly_iso_score_percentile", "sum"),
        anomaly_weeks_ocsvm=("anomaly_ocsvm_score_percentile", "sum"),
        anomaly_weeks_lof=("anomaly_lof_score_percentile", "sum"),
        total_weeks=("week", "nunique"),
        avg_anomaly_score_iso=("iso_score", "mean"),
        avg_anomaly_score_ocsvm=("ocsvm_score", "mean"),
        avg_anomaly_score_lof=("lof_score", "mean"),
    )
    ok = agg["total_weeks"] >= min_distinct_weeks
    tw = agg["total_weeks"].replace(0, np.nan)
    agg["share_anomaly_weeks_iso"] = np.where(
        ok, agg["anomaly_weeks_iso"] / tw, np.nan
    )
    agg["share_anomaly_weeks_ocsvm"] = np.where(
        ok, agg["anomaly_weeks_ocsvm"] / tw, np.nan
    )
    agg["share_anomaly_weeks_lof"] = np.where(
        ok, agg["anomaly_weeks_lof"] / tw, np.nan
    )
    return agg[
        [
            "waiter_id",
            "share_anomaly_weeks_iso",
            "share_anomaly_weeks_ocsvm",
            "share_anomaly_weeks_lof",
            'avg_anomaly_score_iso',
            'avg_anomaly_score_ocsvm',
            'avg_anomaly_score_lof',
        ]
    ]


def _fraud_waiter_ids(df: pd.DataFrame, client_data: pd.DataFrame) -> set:
    cd = client_data
    fraud_waiter_ids = set(
        cd.loc[cd.index.isin(FRAUD_IDS), "top_waiter_id"]
        .dropna()
        .unique()
    )

    # Co-waiters at the same place: 3+ distinct transactions with a fraud person_id.
    fp = df[df["person_id"].isin(FRAUD_IDS)]
    n_by_place_waiter = (
        fp.groupby(["person_id", "place_id", "waiter_id"])["trn_id"]
        .nunique()
        .reset_index(name="n_trn")
    )
    colluders = n_by_place_waiter.loc[
        n_by_place_waiter["n_trn"] >= 3, "waiter_id"
    ].dropna()
    fraud_waiter_ids.update(colluders.unique())

    print("Num of fraud waiter ids: ", len(fraud_waiter_ids))
    return fraud_waiter_ids


def build_waiter_data(df: pd.DataFrame, client_data: pd.DataFrame) -> pd.DataFrame:
    """
    Build waiter-level DataFrame from transaction df and client_data (with anomaly scores).
    client_data must have anomaly_score_iso, anomaly_score_ocsvm, anomaly_score_lof,
    top_waiter_id, num_of_trn, days_visits, is_fraud (index: person_id).
    df must have person_id, waiter_id, place_id, date, trn_id.
    """
    if "anomaly_score_iso" not in df.columns and "anomaly_score_iso" in client_data.columns:
        scores = client_data[["anomaly_score_iso", "anomaly_score_ocsvm", "anomaly_score_lof"]]
        df = df.merge(scores, left_on="person_id", right_index=True, how="left")
    elif "anomaly_score_iso" not in df.columns:
        raise ValueError("Anomaly scores must be in df or client_data")

    waiter_data = df.groupby("waiter_id").agg(
        iso_90=("anomaly_score_iso", lambda x: x.quantile(0.9)),
        ocsvm_90=("anomaly_score_ocsvm", lambda x: x.quantile(0.9)),
        lof_90=("anomaly_score_lof", lambda x: x.quantile(0.9)),
        num_of_trn=("trn_id", "nunique"),
        num_of_clients=("person_id", "nunique"),
        working_days=("date", "nunique"),
    ).reset_index()
    fraud_waiter_ids = _fraud_waiter_ids(df, client_data)
    waiter_data["is_fraud"] = waiter_data["waiter_id"].isin(fraud_waiter_ids)

    active_person_ids = client_data[
        (client_data["num_of_trn"] > 5) & (client_data["days_visits"] > 5)
    ].index

    active_clients_per_waiter = (
        df[df["person_id"].isin(active_person_ids)]
        .groupby("waiter_id")["person_id"]
        .nunique()
        .rename("num_of_active_clients")
    )
    waiter_data = waiter_data.merge(
        active_clients_per_waiter, left_on="waiter_id", right_index=True, how="left"
    )
    waiter_data["num_of_active_clients"] = waiter_data["num_of_active_clients"].fillna(0).astype(int)

    person_place_waiter = (
        df.groupby(["person_id", "place_id"])["waiter_id"]
        .nunique()
        .reset_index(name="num_waiters_in_place")
    )
    single_waiter_person_place = person_place_waiter[
        person_place_waiter["num_waiters_in_place"] == 1
    ][["person_id", "place_id"]]
    person_place_waiter_map = df[["person_id", "place_id", "waiter_id"]].drop_duplicates()
    single_waiter_records = single_waiter_person_place.merge(
        person_place_waiter_map, on=["person_id", "place_id"], how="left"
    )

    clients_only_this_waiter = (
        single_waiter_records.groupby("waiter_id")["person_id"]
        .nunique()
        .rename("num_clients_only_this_waiter")
    )
    waiter_data = waiter_data.merge(
        clients_only_this_waiter, left_on="waiter_id", right_index=True, how="left"
    )
    waiter_data["num_clients_only_this_waiter"] = (
        waiter_data["num_clients_only_this_waiter"].fillna(0).astype(int)
    )

    single_waiter_active_records = single_waiter_records[
        single_waiter_records["person_id"].isin(active_person_ids)
    ]
    active_clients_only_this_waiter = (
        single_waiter_active_records.groupby("waiter_id")["person_id"]
        .nunique()
        .rename("num_active_clients_only_this_waiter")
    )
    waiter_data = waiter_data.merge(
        active_clients_only_this_waiter, left_on="waiter_id", right_index=True, how="left"
    )
    waiter_data["num_active_clients_only_this_waiter"] = (
        waiter_data["num_active_clients_only_this_waiter"].fillna(0).astype(int)
    )

    person_waiter = (
        df.groupby("person_id")["waiter_id"].nunique().reset_index(name="num_waiters_total")
    )
    single_waiter_total_persons = person_waiter[person_waiter["num_waiters_total"] == 1][["person_id"]]
    person_waiter_total_map = df[["person_id", "waiter_id"]].drop_duplicates()
    single_waiter_total_records = single_waiter_total_persons.merge(
        person_waiter_total_map, on="person_id", how="left"
    )

    clients_single_waiter_total = (
        single_waiter_total_records.groupby("waiter_id")["person_id"]
        .nunique()
        .rename("num_clients_single_waiter_total")
    )
    waiter_data = waiter_data.merge(
        clients_single_waiter_total, left_on="waiter_id", right_index=True, how="left"
    )
    waiter_data["num_clients_single_waiter_total"] = (
        waiter_data["num_clients_single_waiter_total"].fillna(0).astype(int)
    )

    single_waiter_total_active_records = single_waiter_total_records[
        single_waiter_total_records["person_id"].isin(active_person_ids)
    ]
    active_clients_single_waiter_total = (
        single_waiter_total_active_records.groupby("waiter_id")["person_id"]
        .nunique()
        .rename("num_active_clients_single_waiter_total")
    )
    waiter_data = waiter_data.merge(
        active_clients_single_waiter_total, left_on="waiter_id", right_index=True, how="left"
    )
    waiter_data["num_active_clients_single_waiter_total"] = (
        waiter_data["num_active_clients_single_waiter_total"].fillna(0).astype(int)
    )

    waiter_data["share_clients_only_this_waiter"] = (
        waiter_data["num_clients_only_this_waiter"] / waiter_data["num_of_clients"]
    ).fillna(0)
    waiter_data["share_active_clients_only_this_waiter"] = (
        waiter_data["num_active_clients_only_this_waiter"]
        / waiter_data["num_of_active_clients"].replace(0, np.nan)
    ).fillna(0)
    waiter_data["share_clients_single_waiter_total"] = (
        waiter_data["num_clients_single_waiter_total"] / waiter_data["num_of_clients"]
    ).fillna(0)
    waiter_data["share_active_clients_single_waiter_total"] = (
        waiter_data["num_active_clients_single_waiter_total"]
        / waiter_data["num_of_active_clients"].replace(0, np.nan)
    ).fillna(0)

    return waiter_data

# Match `config.load_data` client filters (adjust to mirror your modeling pipeline).
ACTIVITY_STATE = 1
DAYS_VISITS = 1

parquet_dir = Path(DATA_PATH)
scores_path = ROOT / "client_anomaly_scores.csv"
out_path = parquet_dir / "waiter_level_features.parquet"

df = pd.read_parquet(parquet_dir / "processed_transactions.parquet", engine="pyarrow")

client_data = pd.read_parquet(parquet_dir / "client_level_features.parquet", engine="pyarrow")
client_data = client_data[
    (client_data["num_of_trn"] > ACTIVITY_STATE) & (client_data["days_visits"] > DAYS_VISITS)
].copy()
client_data["is_fraud"] = client_data["person_id"].isin(FRAUD_IDS).astype(int)
client_data = client_data.set_index("person_id")

score_df = pd.read_csv(scores_path)
score_df = score_df.rename(
    columns={
        "iso_score": "anomaly_score_iso",
        "ocsvm_score": "anomaly_score_ocsvm",
        "lof_score": "anomaly_score_lof",
    }
).set_index("person_id")

client_data = client_data.join(
    score_df[["anomaly_score_iso", "anomaly_score_ocsvm", "anomaly_score_lof"]],
    how="inner",
)
client_data = client_data.dropna(
    subset=["anomaly_score_iso", "anomaly_score_ocsvm", "anomaly_score_lof"]
)

df = df[df["person_id"].isin(client_data.index)].copy()

waiter_level = build_waiter_data(df, client_data)

ww_scores_path = ROOT / "waiter_week_anomaly_scores.csv"
ww_scores = pd.read_csv(ww_scores_path)
share_weeks = _share_anomaly_weeks_from_waiter_week_scores(ww_scores)
waiter_level = waiter_level.merge(share_weeks, on="waiter_id", how="left")


waiter_level.to_parquet(out_path, index=False, engine="pyarrow")
print(f"Wrote {len(waiter_level)} rows to {out_path}")
waiter_level.head()

Num of fraud waiter ids:  30
Wrote 5353 rows to /Users/yuliia/Documents/Fraud-Detection/parquet/waiter_level_features.parquet


,waiter_id,iso_90,ocsvm_90,lof_90,num_of_trn,num_of_clients,working_days,is_fraud,num_of_active_clients,num_clients_only_this_waiter,...,share_clients_only_this_waiter,share_active_clients_only_this_waiter,share_clients_single_waiter_total,share_active_clients_single_waiter_total,share_anomaly_weeks_iso,share_anomaly_weeks_ocsvm,share_anomaly_weeks_lof,avg_anomaly_score_iso,avg_anomaly_score_ocsvm,avg_anomaly_score_lof
0,539f5da6c76b7cd7fa3d859c_0,0.542703,-0.018725,1.160229,5,4,1,False,2,4,...,1.000000,1.000000,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,539f5da6c76b7cd7fa3d859c_0.31,0.526659,-0.031042,1.018016,2,2,2,False,2,0,...,0.000000,0.000000,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,539f5da6c76b7cd7fa3d859c_00,0.525061,-0.033092,1.140944,136,131,25,False,72,81,...,0.618321,0.638889,0.0,0.0,0.142857,0.714286,0.000000,0.438501,0.001436,1.039222
3,539f5da6c76b7cd7fa3d859c_001,0.529399,-0.028704,1.111019,2021,1782,290,False,961,1110,...,0.622896,0.521332,0.0,0.0,0.069444,0.055556,0.069444,0.386409,-0.019145,1.065493
4,539f5da6c76b7cd7fa3d859c_00131,0.491820,-0.082152,1.096395,2,2,2,False,0,1,...,0.500000,0.000000,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
waiter_level

,waiter_id,iso_90,ocsvm_90,lof_90,num_of_trn,num_of_clients,working_days,is_fraud,num_of_active_clients,num_clients_only_this_waiter,num_active_clients_only_this_waiter,num_clients_single_waiter_total,num_active_clients_single_waiter_total,share_clients_only_this_waiter,share_active_clients_only_this_waiter,share_clients_single_waiter_total,share_active_clients_single_waiter_total,share_anomaly_weeks_iso,share_anomaly_weeks_ocsvm,share_anomaly_weeks_lof
0,539f5da6c76b7cd7fa3d859c_0,0.542703,-0.018725,1.160229,5,4,1,False,2,4,2,0,0,1.000000,1.000000,0.0,0.0,NaN,NaN,NaN
1,539f5da6c76b7cd7fa3d859c_0.31,0.526659,-0.031042,1.018016,2,2,2,False,2,0,0,0,0,0.000000,0.000000,0.0,0.0,NaN,NaN,NaN
2,539f5da6c76b7cd7fa3d859c_00,0.525061,-0.033092,1.140944,136,131,25,False,72,81,46,0,0,0.618321,0.638889,0.0,0.0,0.000000,0.142857,0.142857
3,539f5da6c76b7cd7fa3d859c_001,0.529399,-0.028704,1.111019,2021,1782,290,False,961,1110,501,0,0,0.622896,0.521332,0.0,0.0,0.027778,0.041667,0.083333
4,539f5da6c76b7cd7fa3d859c_00131,0.491820,-0.082152,1.096395,2,2,2,False,0,1,0,0,0,0.500000,0.000000,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5348,675ac8c769a6f18612504814_17494,0.536910,-0.041957,1.160229,90,72,7,False,50,65,43,0,0,0.902778,0.860000,0.0,0.0,0.000000,0.000000,0.000000
5349,675ac8c769a6f18612504814_18352,0.516226,-0.047856,1.140218,79,74,5,False,54,68,48,0,0,0.918919,0.888889,0.0,0.0,0.000000,0.000000,0.000000
5350,675ac8c769a6f18612504814_19362,0.543888,-0.010709,1.101054,22,19,2,False,14,17,12,0,0,0.894737,0.857143,0.0,0.0,0.000000,0.000000,0.000000
5351,6762deb569a624876c17774a_acherepania,0.533562,-0.062160,1.167703,1,1,1,False,0,1,0,0,0,1.000000,0.000000,0.0,0.0,NaN,NaN,NaN
